# Section 2.2 — Spherical Harmonics (Weinberg §2.2)

**What this section claims**
- The eigenfunctions of L² (eigenvalue ℏ²l(l+1)) and L₃ (ℏm) on the unit sphere are the spherical harmonics Y_l^m(θ,φ).
- Y_l^m(θ,φ) = N_l^m P_l^m(cosθ) e^{imφ} with N_l^m = √[(2l+1)(l−m)!/(4π(l+m)!)] — the Y's are orthonormal on the sphere.
- They solve the associated Legendre equation; regularity at θ = 0, π forces l = 0,1,2,… and integer m with |m| ≤ l (the algebraic result of §2.1 re-derived from a differential equation).
- Parity: Y_l^m(π−θ, φ+π) = (−1)^l Y_l^m(θ,φ); complex conjugation: (Y_l^m)* = (−1)^m Y_l^{−m}.


## Associated Legendre equation from L²Y = ℏ²l(l+1)Y (symbolic)

Writing Y = Θ(θ)e^{imφ} and using the spherical form of L² turns the eigenvalue equation into the associated Legendre equation for x = cosθ. Verify that sympy's assoc_legendre satisfies it:


In [ ]:
import sympy as sp

x = sp.symbols('x', real=True)
for lv in range(4):
    for mv in range(lv+1):
        P = sp.assoc_legendre(lv, mv, x)
        ode = sp.diff((1-x**2)*sp.diff(P, x), x) + (lv*(lv+1) - mv**2/(1-x**2))*P
        print(f"l={lv}, m={mv}: residual of associated Legendre eq = {sp.simplify(ode)}")

## Normalization and orthonormality (symbolic + numeric)


In [ ]:
# integral identity:  int_{-1}^{1} [P_l^m(x)]^2 dx = 2/(2l+1) * (l+m)!/(l-m)!
for lv in range(4):
    for mv in range(lv+1):
        I = sp.integrate(sp.assoc_legendre(lv, mv, x)**2, (x, -1, 1))
        print(f"l={lv}, m={mv}: integral = {sp.simplify(I)}   formula: {sp.Rational(2,2*lv+1)*sp.factorial(lv+mv)/sp.factorial(lv-mv)}")

In [ ]:
# full spherical-harmonic orthonormality:  int dOmega Y_lm* Y_l'm' = delta_ll' delta_mm'
# quadrature: Gauss-Legendre in x = cos theta (exact in theta), uniform grid in phi (exact for |m-m'| < 64)
import numpy as np
from scipy.special import sph_harm
from numpy.polynomial.legendre import leggauss

xg, wg = leggauss(32)
TH = np.arccos(xg)
PH = 2*np.pi*np.arange(64)/64
ls = [(l, m) for l in range(3) for m in range(-l, l+1)]
Y = np.array([sph_harm(m, l, PH[:, None], TH[None, :]) for (l, m) in ls])   # shape (n_sh, n_phi, n_theta)
G = np.einsum('aft,bft,t->ab', np.conj(Y), Y, wg) * (2*np.pi/64)
print("max |Gram - identity| over the l<=2 shell:", np.abs(G - np.eye(len(ls))).max())

## Parity and conjugation (numeric check on random angles)


In [ ]:
import numpy as np
rng = np.random.default_rng(7)
worst_parity, worst_conj = 0.0, 0.0
for l in range(4):
    for m in range(-l, l+1):
        for _ in range(20):
            th, ph = rng.uniform(0, np.pi), rng.uniform(0, 2*np.pi)
            Y1 = sph_harm(m, l, ph, th)                       # forward
            Y2 = sph_harm(m, l, ph+np.pi, np.pi-th)            # parity-reversed point
            worst_parity = max(worst_parity, abs(Y2 - (-1)**l*Y1))
            worst_conj = max(worst_conj, abs(np.conj(Y1) - (-1)**m*sph_harm(-m, l, ph, th)))
print(f"max |Y(pi-theta, phi+pi) - (-1)^l Y| = {worst_parity:.2e}")
print(f"max |Y* - (-1)^m Y_l^(-m)|        = {worst_conj:.2e}")

## Numerics: |Y_l^m|² on the sphere for l = 1, 2


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa
th = np.linspace(0, np.pi, 100); ph = np.linspace(0, 2*np.pi, 100)
TH, PH = np.meshgrid(th, ph)
fig = plt.figure(figsize=(11, 3.6))
for i, (l, m) in enumerate([(1, 0), (2, 0), (2, 2)]):
    Rr = np.abs(sph_harm(m, l, PH, TH))**2
    xs, ys, zs = Rr*np.sin(TH)*np.cos(PH), Rr*np.sin(TH)*np.sin(PH), Rr*np.cos(TH)
    ax = fig.add_subplot(1, 3, i+1, projection='3d')
    ax.plot_surface(xs, ys, zs, rstride=2, cstride=2, cmap='viridis', lw=0)
    ax.set_title(f'l={l}, m={m}', fontsize=10); ax.set_axis_off()
plt.tight_layout(); plt.show()
print("z-axis lobes for m=0; torus for |m|=2 -- rotational nodes at theta = pi/2")

## My summary
*(fill in — own words, ≤ 5 sentences)*

## Gaps
*(questions to raise in review)*
